# Human-in-the-loop regulatory checkpoint

This standalone prototype takes an upstream `industry` and `insight`, plus an explicit jurisdiction. It uses Gemini Search grounding, validates a structured advisory assessment, and then asks a human to **approve, revise, or reject**. It is not yet connected to the local arXiv retrieval pipeline.

From the repository root, install `requirements.txt` in the notebook kernel environment, copy `.env.example` to `.env`, and set your own `GOOGLE_API_KEY`. Never commit `.env` or notebook outputs containing private insights. Model calls use your account quota. `GOOGLE_MODEL` must support Search grounding and structured output (default: `gemini-2.5-flash`). See `docs/human-checkpoint.md` for the output contract and troubleshooting.


In [ ]:
import json
import sys
from pathlib import Path

# Supports a kernel launched from the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "src" / "human_checkpoint.py").is_file():
    repo_root = repo_root.parent
if not (repo_root / "src" / "human_checkpoint.py").is_file():
    raise RuntimeError("Launch this notebook from the repository root or notebooks/.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.human_checkpoint import (
    create_client, retrieve_regulations, assess_insight, record_human_review,
)

client = create_client()


## Input from the insight generator
Replace the example with the actual upstream output. Jurisdiction cannot be inferred from industry; set it for the intended deployment. The proposal below is deliberately unverified, not a claim of compliance.


In [ ]:
insight_data = {
    "industry": "finance",
    "insight": "Banks can fully automate quarterly reporting.",
    "jurisdiction": "United States (federal banking regulation)",
}


## Retrieve cited regulatory evidence
This invokes the Google Search tool, not an existing regulation knowledge base. Unsupported generation or missing citation segments stops execution. Search citations do not prove that a source is authoritative or that a rule applies: the reviewer must open the sources and check jurisdiction, dates, and applicability.


In [ ]:
# Clear earlier results first so a failed rerun cannot leave an old approval active.
regulatory_evidence = None
checkpoint = None
reviewed_checkpoint = None
regulatory_evidence = retrieve_regulations(client, **insight_data)
print(json.dumps(regulatory_evidence, indent=2))


## Structured advisory assessment
A separate Gemini call uses only the cited evidence and the `Assessment` schema. Unknown source IDs, missing citations, and contradictory verdicts are rejected. Missing business/operational context must remain unknown. The result always starts with `approved: false` and a pending human review.


In [ ]:
checkpoint = None
reviewed_checkpoint = None
checkpoint = assess_insight(client, regulatory_evidence)
print(json.dumps(checkpoint, indent=2))


## Explicit human checkpoint
Inspect each linked source and each finding before answering. Record whether sources are official, current, and applicable, and what business evidence supports feasibility. Approval is blocked for `conflicts` or `insufficient_evidence`; revise the proposal or gather missing evidence and rerun instead. This console review is a prototype audit record, not an authenticated production approval service.


In [ ]:
reviewed_checkpoint = None
if checkpoint is None:
    raise RuntimeError("Run a successful assessment before human review.")
decision = input("Decision (approve / revise / reject): ").strip().lower()
reviewer = input("Reviewer name: ").strip()
notes = input("Review notes (source checks, applicability, feasibility): ").strip()
reviewed_checkpoint = record_human_review(
    checkpoint, decision=decision, reviewer=reviewer, notes=notes,
)
print(json.dumps(reviewed_checkpoint, indent=2))


Downstream code must consume `reviewed_checkpoint` and check `approved is True` before proceeding. A model verdict is never approval. Saving or forwarding the result is intentionally not automatic; keep sensitive output local and clear notebook outputs before committing.
